# Task 4: Custom Gradient Descent Optimizers with Momentum, RMSprop, and Adam

**Objective:** Master adaptive learning rate mechanics and temporal update optimization algorithms by implementing high-performance weight updates from scratch using raw PyTorch tensor math (no `torch.optim`).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

class CustomOptimizer:
    def __init__(self, params):
        self.params = list(params)
        
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

class CustomSGDMomentum(CustomOptimizer):
    def __init__(self, params, lr=0.01, beta=0.9):
        super().__init__(params)
        self.lr = lr
        self.beta = beta
        self.velocities = [torch.zeros_like(p) for p in self.params]
        
    def step(self):
        for p, v in zip(self.params, self.velocities):
            if p.grad is None: continue
            # Update velocity
            v.mul_(self.beta).add_(p.grad, alpha=1 - self.beta)
            # Update parameter
            p.add_(v, alpha=-self.lr)

class CustomRMSprop(CustomOptimizer):
    def __init__(self, params, lr=0.001, alpha=0.99, eps=1e-8):
        super().__init__(params)
        self.lr = lr
        self.alpha = alpha
        self.eps = eps
        self.squared_grads = [torch.zeros_like(p) for p in self.params]
        
    def step(self):
        for p, s in zip(self.params, self.squared_grads):
            if p.grad is None: continue
            # Moving average of squared gradients
            s.mul_(self.alpha).addcmul_(p.grad, p.grad, value=1 - self.alpha)
            # Param updates
            denom = s.sqrt().add_(self.eps)
            p.addcdiv_(p.grad, denom, value=-self.lr)

class CustomAdam(CustomOptimizer):
    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8):
        super().__init__(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0
        
    def step(self):
        self.t += 1
        for p, m_prev, v_prev in zip(self.params, self.m, self.v):
            if p.grad is None: continue
            
            # First moment update
            m_prev.mul_(self.beta1).add_(p.grad, alpha=1 - self.beta1)
            
            # Second moment update
            v_prev.mul_(self.beta2).addcmul_(p.grad, p.grad, value=1 - self.beta2)
            
            # Bias corrections
            bias_corr1 = 1 - self.beta1 ** self.t
            bias_corr2 = 1 - self.beta2 ** self.t
            
            step_size = self.lr / bias_corr1
            denom = (v_prev.sqrt() / np.sqrt(bias_corr2)).add_(self.eps)
            
            # Update step
            p.addcdiv_(m_prev, denom, value=-step_size)

In [ ]:
# Create synthetic double-moon dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

def build_model():
    # Simple MLP classifier
    return nn.Sequential(
        nn.Linear(2, 32),
        nn.ReLU(),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Linear(16, 2)
    )

epochs = 200
criterion = nn.CrossEntropyLoss()

optimizers_to_test = [
    ("SGD with Momentum", lambda p: CustomSGDMomentum(p, lr=0.1, beta=0.9)),
    ("RMSprop", lambda p: CustomRMSprop(p, lr=0.01, alpha=0.99)),
    ("Adam", lambda p: CustomAdam(p, lr=0.01, betas=(0.9, 0.999)))
]

losses_history = {}

for name, opt_builder in optimizers_to_test:
    model = build_model()
    optimizer = opt_builder(model.parameters())
    losses_history[name] = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        
        losses_history[name].append(loss.item())
        
        if (epoch + 1) % 50 == 0:
            print(f"{name:18s} | Epoch {epoch+1:3d}/{epochs} | Loss: {loss.item():.4f}")

# Visualizing convergence
plt.figure(figsize=(10, 6))
for name, losses in losses_history.items():
    plt.plot(losses, label=name, lw=2.5)
plt.title('Loss Convergence Profiles across Optimizers')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True)
plt.legend()
plt.show()